In [10]:
import json
import logging
from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional

# 配置全局日志
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("ToolSystem")


# ==============================================================================
# 1. Tool 基类与具体工具实现
# ==============================================================================


class Tool(ABC):
    """Tool 抽象基类"""

    def __init__(
        self,
        name: str,
        description: str,
        required_params: Optional[List[str]] = None,
    ):
        self.name = name
        self.description = description
        self.required_params = required_params or []

    @abstractmethod
    def execute(self, **kwargs) -> Any:
        pass


class EmployeeTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_employee_info",
            description="根据员工 ID 查询员工基本信息",
            required_params=["employee_id"],
        )

    def execute(self, employee_id: str, **kwargs) -> dict:
        if employee_id == "E1001":
            return {
                "employee_id": "E1001",
                "name": "Kevin",
                "dept": "Corp.IT",
                "role": "Developer",
            }
        raise ValueError(f"未找到 ID 为 '{employee_id}' 的员工数据")


class WeatherTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_weather",
            description="查询指定城市的天气",
            required_params=["city"],
        )

    def execute(self, city: str, **kwargs) -> dict:
        mock_weather = {
            "上海": {"temperature": "28°C", "condition": "多云"},
            "北京": {"temperature": "25°C", "condition": "晴"},
        }
        if city in mock_weather:
            return {"city": city, **mock_weather[city]}
        return {"city": city, "temperature": "22°C", "condition": "未知"}


class OrderTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_order_details",
            description="根据订单号查询订单状态",
            required_params=["order_id"],
        )

    def execute(self, order_id: str, **kwargs) -> dict:
        if order_id.startswith("ORD"):
            return {
                "order_id": order_id,
                "status": "COMPLETED",
                "amount": "$20.00",
            }
        raise ValueError(f"无效的订单编号格式: '{order_id}'")


# ==============================================================================
# 2. ToolRegistry (注册表)
# ==============================================================================


class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}

    def register(self, tool: Tool) -> None:
        if tool.name in self._tools:
            logger.warning(
                f"[Registry] 工具 '{tool.name}' 已存在，将被覆盖。"
            )
        self._tools[tool.name] = tool
        logger.info(f"[Registry] 成功注册工具: [{tool.name}]")

    def get_tool(self, name: str) -> Optional[Tool]:
        return self._tools.get(name)

    def list_tools(self) -> Dict[str, str]:
        return {name: tool.description for name, tool in self._tools.items()}


# ==============================================================================
# 3. ToolManager (新增 tool_call 方法)
# ==============================================================================


class ToolManager:
    """负责校验、执行、异常处理、日志输出"""

    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def _validate_args(self, tool: Tool, kwargs: dict) -> None:
        missing = [
            param
            for param in tool.required_params
            if param not in kwargs or kwargs[param] is None
        ]
        if missing:
            raise ValueError(
                f"工具 [{tool.name}] 缺少必要参数: {', '.join(missing)}"
            )

    def invoke(self, tool_name: str, **kwargs) -> dict:
        """底层标准执行方法"""
        logger.info(
            f"[Manager] ==> 请求调用工具: [{tool_name}] | 参数: {kwargs}"
        )

        tool = self.registry.get_tool(tool_name)
        if not tool:
            err_msg = f"未找到名为 '{tool_name}' 的工具"
            logger.error(f"[Manager] <== 执行失败: {err_msg}")
            return {"success": False, "error": err_msg, "data": None}

        try:
            self._validate_args(tool, kwargs)
            result = tool.execute(**kwargs)
            logger.info(
                f"[Manager] <== 工具 [{tool_name}] 执行成功 | 结果: {result}"
            )
            return {"success": True, "error": None, "data": result}
        except Exception as e:
            logger.error(
                f"[Manager] <== 工具 [{tool_name}] 执行过程中抛出异常: {str(e)}"
            )
            return {"success": False, "error": str(e), "data": None}

    def tool_call(self, call_data: dict) -> dict:
        """
        新增：直接处理大模型返回的 Tool Call 结构

        支持的输入格式：
        1. 自定义格式: {"tool": "get_weather", "arguments": {"city": "上海"}}
        2. OpenAI 格式: {"name": "get_weather", "arguments": "{\"city\": \"上海\"}"}
        """
        if not isinstance(call_data, dict):
            return {
                "success": False,
                "error": "tool_call 请求参数必须是字典类型",
                "data": None,
            }

        # 兼容 "tool" 或 "name" 键名
        tool_name = call_data.get("tool") or call_data.get("name")
        arguments = call_data.get("arguments", {})

        if not tool_name:
            return {
                "success": False,
                "error": "tool_call 结构中缺失工具名称 ('tool' 或 'name')",
                "data": None,
            }

        # 如果大模型返回的 arguments 是 JSON 格式的字符串，自动将其解析为字典
        if isinstance(arguments, str):
            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError as e:
                return {
                    "success": False,
                    "error": f"arguments 解析 JSON 失败: {str(e)}",
                    "data": None,
                }

        # 转交给基础的 invoke 执行
        return self.invoke(tool_name, **arguments)


# ==============================================================================
# 4. 演示运行 (Demo)
# ==============================================================================

if __name__ == "__main__":
    registry = ToolRegistry()
    manager = ToolManager(registry)

    # 注册工具
    registry.register(EmployeeTool())
    registry.register(WeatherTool())
    registry.register(OrderTool())

    print("\n" + "=" * 50)

    # --- 演示 1: 直接使用自定义字典调用 tool_call 方法 ---
    call_1 = {"tool": "get_weather", "arguments": {"city": "上海"}}
    res1 = manager.tool_call(call_1)
    print("调用结果 1:", res1)
    print("=" * 50)

    # --- 演示 2: 兼容 OpenAI 格式（"name" + JSON 格式字符串 arguments） ---
    call_2 = {
        "name": "get_employee_info",
        "arguments": '{"employee_id": "E1001"}',  # JSON 字符串形式
    }
    res2 = manager.tool_call(call_2)
    print("调用结果 2:", res2)
    print("=" * 50)

    # --- 演示 3: 异常调用（缺少必要参数） ---
    call_3 = {"tool": "get_weather", "arguments": {}}
    res3 = manager.tool_call(call_3)
    print("调用结果 3:", res3)

[2026-07-24 16:51:01] [INFO] [Registry] 成功注册工具: [get_employee_info]
[2026-07-24 16:51:01] [INFO] [Registry] 成功注册工具: [get_weather]
[2026-07-24 16:51:01] [INFO] [Registry] 成功注册工具: [get_order_details]
[2026-07-24 16:51:01] [INFO] [Manager] ==> 请求调用工具: [get_weather] | 参数: {'city': '上海'}
[2026-07-24 16:51:01] [INFO] [Manager] <== 工具 [get_weather] 执行成功 | 结果: {'city': '上海', 'temperature': '28°C', 'condition': '多云'}
[2026-07-24 16:51:01] [INFO] [Manager] ==> 请求调用工具: [get_employee_info] | 参数: {'employee_id': 'E1001'}
[2026-07-24 16:51:01] [INFO] [Manager] <== 工具 [get_employee_info] 执行成功 | 结果: {'employee_id': 'E1001', 'name': 'Kevin', 'dept': 'Corp.IT', 'role': 'Developer'}
[2026-07-24 16:51:01] [INFO] [Manager] ==> 请求调用工具: [get_weather] | 参数: {}
[2026-07-24 16:51:01] [ERROR] [Manager] <== 工具 [get_weather] 执行过程中抛出异常: 工具 [get_weather] 缺少必要参数: city



调用结果 1: {'success': True, 'error': None, 'data': {'city': '上海', 'temperature': '28°C', 'condition': '多云'}}
调用结果 2: {'success': True, 'error': None, 'data': {'employee_id': 'E1001', 'name': 'Kevin', 'dept': 'Corp.IT', 'role': 'Developer'}}
调用结果 3: {'success': False, 'error': '工具 [get_weather] 缺少必要参数: city', 'data': None}
